# Task B -- full-data context-tagged submission

This notebook trains the context-tagged version of the current best Task B recipe
on every labelled row and writes a CodaBench submission ZIP. It uses TAPT MuRIL,
one-layer reinitialization, R-Drop and the fold-safe context tagging code from
Experiment 11.

| setting | value |
|---|---|
| TAPT | all 6,406 permitted comments, no holdout or deduplication |
| classifier | all 3,159 labelled Task B rows, no deduplication |
| context tags | enabled; gazetteer fitted on all labelled rows |
| reinitialization | one final MuRIL encoder layer |
| R-Drop | 0.5 |
| seeds | 42, 43, 44, 45, 46; validation probabilities averaged |

There is no honest local F1 for this run because every labelled row is used for
training. The output ZIP is the artifact to submit to the Task B validation phase.
Expected runtime is about 2--3 hours on a T4 x2 or P100. Use GPU, Internet, and
**Save Version -> Save & Run All**.

In [ ]:
import os, pathlib, re, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Build or reuse full-data TAPT

This is the same permitted TAPT corpus used by the current best submission. No
labelled or unlabelled comments are held out at this stage.

In [ ]:
TAPT_OUT = "artifacts/runs/tapt-ctxdecode"
TAPT_LOG = "artifacts/logs/context_tags_full_tapt.log"
if (pathlib.Path(TAPT_OUT) / "config.json").exists():
    print("using existing TAPT checkpoint:", TAPT_OUT)
else:
    run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
         "--corpus", "data/raw/multiclass_train.csv", "data/external/offenseval_kn.csv",
         "--val-frac", "0", "--min-words", "1", "--no-dedupe",
         "--epochs", "8", "--out", TAPT_OUT], log=TAPT_LOG)
assert (pathlib.Path(TAPT_OUT) / "config.json").exists(), "TAPT checkpoint was not written"
print("TAPT checkpoint ready:", TAPT_OUT)

## 2. Train five full-data context-tagged models

`--folds 1` means every seed trains on all 3,159 labelled rows. With full-data
training there is no validation split and therefore no local score. The topic
gazetteer is fitted from all labelled rows, which is appropriate for this final fit.

In [ ]:
TAG = "b_context_tags_full"
TRAIN_LOG = pathlib.Path("artifacts/logs") / f"{TAG}.log"
RUN_DIR = pathlib.Path("artifacts/runs") / TAG
run([sys.executable, "-u", "-m", "hastika.task_b.train",
     "--tag", TAG,
     "--model", TAPT_OUT,
     "--folds", "1",
     "--no-dedupe",
     "--tags",
     "--reinit-layers", "1",
     "--rdrop", "0.5",
     "--aux-weight", "0",
     "--seeds", "42", "43", "44", "45", "46",
     "--epochs", "6"], log=str(TRAIN_LOG))
log_text = TRAIN_LOG.read_text()
fits = re.findall(r"===== seed (\d+) FULL FIT, (\d+) rows, no validation =====", log_text)
assert [seed for seed, _ in fits] == ["42", "43", "44", "45", "46"], fits
assert all(rows == "3159" for _, rows in fits), fits
assert "tags=True" in log_text, "context tags were not enabled"
assert "rdrop=0.5" in log_text, "R-Drop was not set to 0.5"
assert "reinit=1" in log_text, "one-layer reinitialization was not enabled"
assert (RUN_DIR / "test_probs.npy").exists(), "validation probabilities were not written"
assert (RUN_DIR / "predictions.csv").exists(), "predictions were not written"
print("five full-data context-tagged fits completed:", RUN_DIR)

## 3. Validate and package the CodaBench submission

The helper checks the exact `id,label` schema, 395 validation IDs and six allowed
labels, then writes a ZIP containing one bare `predictions.csv`.

In [ ]:
import pandas as pd

PRED = RUN_DIR / "predictions.csv"
ZIP = pathlib.Path("/kaggle/working/b_context_tags_full.zip")
run([sys.executable, "-m", "hastika.common.submission",
     "--task", "b", "--pred", str(PRED), "--out", str(ZIP)])
assert ZIP.exists(), "submission ZIP was not written"
with zipfile.ZipFile(ZIP) as z:
    assert z.namelist() == ["predictions.csv"], z.namelist()
pred = pd.read_csv(PRED)
assert list(pred.columns) == ["id", "label"]
assert len(pred) == 395 and pred["id"].is_unique
print("READY TO UPLOAD:", ZIP)

## 4. Preserve reproducibility files

Download the ZIP and this output folder from Kaggle. Submit only
`b_context_tags_full.zip` to the Task B validation phase.

In [ ]:
OUT = pathlib.Path("/kaggle/working/context_tags_full_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for source in [ZIP, PRED, RUN_DIR / "test_probs.npy", TRAIN_LOG, pathlib.Path(TAPT_LOG)]:
    if source.exists():
        shutil.copy2(source, OUT / source.name)
print("download:", OUT)
print("files:", sorted(x.name for x in OUT.iterdir()))